# Fit a root architecture in five trials

Use this invented example to learn the same whole-root search available in the
Dash **Architecture Search** page. Select the repository's Python 3.10 kernel
and run the cells in order. Each run saves its own outputs locally.

The target is three primary roots in a configuration with maximum order 1.
The field notebook in this folder covers a different task with physical core
measurements and a local study bundle.

In [ ]:
# Search settings
import csv
import json
import os
from pathlib import Path
import subprocess
import sys
from tempfile import mkdtemp

import pandas as pd

from deeprootgen.research.architecture_search import (
    describe_controls,
    run_architecture_search,
    validate_architecture_search,
)

def find_project():
    project = os.environ.get("DRG_PROJECT_ROOT")
    if project:
        return Path(project).resolve()
    here = Path.cwd()
    for candidate in (here, *here.parents):
        if (candidate / "examples/synthetic_minimal.json").is_file():
            return candidate
    raise FileNotFoundError("Open the notebook within a DeepRootGen checkout.")

ROOT = find_project()
BASE_PATH = ROOT / "examples/synthetic_minimal.json"
TARGETS_PATH = ROOT / "examples/research/optimisation_summary_synthetic.csv"
METHOD = "tpe"  # Change to "latin_hypercube" to repeat with fixed sampling.
METRIC = "mae"
TRIALS = 5
STARTUP = 2
SEEDS = [0, 1]
RANGES = [{"control": "primary_root_count", "lower": 2, "upper": 4}]
OUTPUT = Path(os.environ.get("DRG_LESSON_OUTPUT", str(ROOT / ".runtime")))

## Preview the settings

The schema-4 JSON sets the model and active laws. Vary one supported control
here and keep the base values of the others. The target CSV contains a
numeric whole-root statistic. The preview validates the complete request.

In [ ]:
base = json.loads(BASE_PATH.read_text())
with TARGETS_PATH.open(newline="") as handle:
    targets = list(csv.DictReader(handle))
request = {
    "base": base,
    "ranges": RANGES,
    "targets": targets,
    "metric": METRIC,
    "trials": TRIALS,
    "startup": STARTUP,
    "seeds": SEEDS,
    "method": METHOD,
}
job, _, controls, _, _ = validate_architecture_search(request)
for row in RANGES:
    key = row["control"]
    print(f"{controls[key]['label']}: {row['lower']} to {row['upper']}")
print(f"{job.trials} trials by {len(job.seeds)} seeds using {job.method}")
print(f"Target: {targets[0]['statistic_name']} = {targets[0]['statistic_value']}")

## Run the search and inspect the trials

A lower score is better. The expected outcome with the supplied settings is
five completed trials, zero loss and a primary root count of three. Optuna
TPE uses earlier trials to guide proposals after startup. Latin hypercube
spreads proposals through the range and uses the same scoring and seeds.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
lesson = Path(mkdtemp(prefix="architecture-quickstart-", dir=OUTPUT))
fit = lesson / "fit"
summary = run_architecture_search(request, fit)
print(f"{summary['completed_trials']} completed, {summary['failed_trials']} failed")
print(f"Best loss: {summary['best_loss']}")
print(f"Outputs: {lesson}")
pd.read_csv(fit / "trials.csv")

## Replay the best configuration

The saved best configuration uses the first simulation seed. Replay it
to inspect the graph for one realization. The fitted cost used each
configured seed. Open the exported `roots.html` to rotate the graph.

In [ ]:
best_path = fit / "best-configuration.json"
best, active, _ = describe_controls(json.loads(best_path.read_text()))
print(f"Primary count: {active['primary_root_count']['value']}")
print(f"Replay seed: {best['random_seed']}")
replay = lesson / "replay"
subprocess.run(
    [sys.executable, "-m", "deeprootgen", "simulate", "--config", str(best_path),
     "--outdir", str(replay)],
    cwd=ROOT,
    stdout=subprocess.DEVNULL,
    check=True,
)
print(f"Open {replay / 'roots.html'}")

## Try the same settings in Dash

Upload the same base JSON in **Architecture Search**, enter both bounds for
primary root count, and paste the target CSV. Use **Check setup** and
**Run search**. Use **Core Comparison** to score already projected held-core
predictions.

See the [print guide](../../docs/DEEPROOTGEN_QUICKSTART.md) and
[web instructions](../../docs/WEB_OPTIMISATION.md).